Unzip the datasets and familiarize yourself with the data format.

In [1]:
import os
import re
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, precision_recall_fscore_support

In [6]:
DATA_DIR = Path("spamassassin")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://spamassassin.apache.org/old/publiccorpus/"
FILES = {
    "spam": "20021010_spam.tar.bz2",
    "ham": "20021010_easy_ham.tar.bz2"
}

for label, filename in FILES.items():
    file_path = DATA_DIR / filename
    if not file_path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(BASE_URL + filename, file_path)


In [7]:
for filename in FILES.values():
    file_path = DATA_DIR / filename
    with tarfile.open(file_path, "r:bz2") as tar:
        tar.extractall(path=DATA_DIR)

In [8]:
def load_emails_from_dir(path):
    emails = []
    for filename in os.listdir(path):
        if filename.startswith("."):
            continue
        with open(os.path.join(path, filename), encoding="latin1") as f:
            emails.append(f.read())
    return emails

spam_emails = load_emails_from_dir(DATA_DIR / "spam")
ham_emails = load_emails_from_dir(DATA_DIR / "easy_ham")

len(spam_emails), len(ham_emails)


(501, 2551)

Split the data into a training set and a test set

In [10]:
X = spam_emails + ham_emails
y = np.array([1] * len(spam_emails) + [0] * len(ham_emails))  # 1=spam, 0=ham

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Write a data preparation pipeline to convert each email into a feature vector. Your preparation pipeline should transform an email into a (sparse) vector that indicates the presence or absence of each possible word. For example, if all emails only ever contain four words, “Hello”, “how”, “are”, “you”, then the email “Hello you Hello Hello you” would be converted into a vector [1, 0, 0, 1] (meaning [“Hello” is present, “how” is absent, “are” is absent, “you” is present]), or [3, 0, 0, 2] if you prefer to count the number of occurrences of each word.

You may want to add hyperparameters to your preparation pipeline to control whether or not to strip off email headers, convert each email to lowercase, remove punctuation, replace all URLs with “URL”, replace all numbers with “NUMBER”, or even perform stemming (i.e., trim off word endings; there are Python libraries available to do this).

In [11]:
def email_preprocessor(
    text,
    strip_headers=True,
    lowercase=True,
    replace_urls=True,
    replace_numbers=True,
    remove_punctuation=True,
):
    if strip_headers:
        text = re.split(r"\n\n", text, maxsplit=1)[-1]

    if lowercase:
        text = text.lower()

    if replace_urls:
        text = re.sub(r"http\S+|www\.\S+", " URL ", text)

    if replace_numbers:
        text = re.sub(r"\d+", " NUMBER ", text)

    if remove_punctuation:
        text = re.sub(r"[^\w\s]", " ", text)

    return text

In [12]:
vectorizer = CountVectorizer(
    preprocessor=email_preprocessor,
    stop_words="english",
    binary=True,          # set False to count occurrences instead of presence
    min_df=2              # to ignore very rare words
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

X_train_vec.shape

(2441, 23691)

Finally, try out several classifiers and see if you can build a great spam classifier, with both high recall and high precision.

In [19]:
nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

y_pred = nb.predict(X_test_vec)

In [17]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

In [22]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print()

precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.4f}")

recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.4f}")

f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.4f}")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Confusion Matrix:
[[511   0]
 [ 13  87]]

Precision: 1.0000
Recall: 0.8700
F1 Score: 0.9305
Accuracy: 0.9787


In [23]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_vec, y_train)

y_pred = log_reg.predict(X_test_vec)

In [24]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print()

precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.4f}")

recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.4f}")

f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.4f}")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Confusion Matrix:
[[511   0]
 [  8  92]]

Precision: 1.0000
Recall: 0.9200
F1 Score: 0.9583
Accuracy: 0.9869


In [25]:
svm = LinearSVC()
svm.fit(X_train_vec, y_train)

y_pred = svm.predict(X_test_vec)
print(classification_report(y_test, y_pred_svm, target_names=["ham", "spam"]))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       511
        spam       1.00      0.94      0.97       100

    accuracy                           0.99       611
   macro avg       0.99      0.97      0.98       611
weighted avg       0.99      0.99      0.99       611



C:\Users\twolf\anaconda3\lib\site-packages\sklearn\svm\_base.py:1208: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  ConvergenceWarning,


In [26]:
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print()

precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.4f}")

recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.4f}")

f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1:.4f}")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Confusion Matrix:
[[511   0]
 [  6  94]]

Precision: 1.0000
Recall: 0.9400
F1 Score: 0.9691
Accuracy: 0.9902


## Summary of Findings:

I struggled with uploading the data correctly and had to look up a different way to do it. I also got online help and some AI help with the preparation pipeline. Once I got to trying out the classifiers, I looked which classifiers would work well for spam detection, and decided to try Naive Bayes, Logistic Regression, and Linear SVC. Linear SVC was the most accurate of the three, all three had precision of 1.0. I did not have time to do very much evaluation.